# 05 · Feature lists for the two models

One shared candidate list for both models, then some extra trimming for Elastic net only.

- A. Shared candidates: drop columns no model should use (leakage, almost one value). No models are fit
  here.
- B. Elastic net: keep one form of each duplicate and remove high VIF, so the coefficients stay readable. Its
  L1 penalty takes care of weak columns while it fits.
- C. Random forest: uses the shared list as is. Trees don't mind correlated or unused columns (trimming its
  list in notebook 05 changed MAE by at most 0.001). Permutation importance is only used to explain it.

Both models get the same information, so if they disagree on a listing, it's because of the model, not the
inputs.

Setup: 2026-09 snapshots, GroupKFold(5) by host, MAE on log price. 

## A0. Setup

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="Pandas requires version")  # Anaconda 'bottleneck' noise
warnings.filterwarnings("ignore", message=".*not a team snapshot.*")
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "strfair").is_dir())))
import strfair

USE_SENTIMENT = False       # True: strfair.prepare() with Kandice's sentiment features (slow the first time)

SNAPSHOTS = {"new-york-city": "2026-09-14",
             "los-angeles": "2026-09-14",
             "toronto": "2026-09-16",
             "san-diego": "2026-09-20",
             "clark-county-nv": "2026-09-20",
             "jersey-city": "2026-09-21"}

OUT = strfair.REPO_ROOT / "docs" / "eda" / "feature_lists"   # every table is saved here
OUT.mkdir(parents=True, exist_ok=True)

# same as 04_baseline. sentiment_review_count is a review count under another name, so it is never a candidate
drop = strfair.LEAKY + strfair.DEMAND_RESPONSE + strfair.ID_COLUMNS + ["neighbourhood_raw", "discount_amount",
                                                                         "sentiment_review_count"]

# quote controls: they say *when* and *for how long* the quote was taken, so they stay in both models
QUOTE_CONTROLS = ["is_monthly_quote", "quote_nights", "lead_band", "min_nights_band", "weekend_checkin",
                  "checkin_dow", "checkin_month"]

In [2]:
from strfair.data import load_listings
from strfair.prep import _sample_mask


def load_city(city, snapshot):
    if USE_SENTIMENT:
        data = strfair.prepare(city, snapshot)
        sample, F = data.sample, data.features
    else:
        # same rows and features as strfair.prepare(), without the sentiment step and without the cache
        listings = load_listings(city, snapshot)
        sample = listings[_sample_mask(listings)].reset_index(drop=True)
        F = strfair.build_features(sample, city).reset_index(drop=True)

    X = F[[c for c in F.columns if c not in drop]].copy()
    if "avg_sentiment" in X:
        X["no_sentiment"] = X["avg_sentiment"].isna().astype(int)   # 1 = fewer than 3 usable reviews
    y = F["log_price"]
    groups = sample["host_id"].values                               # one host's listings stay in the same fold
    return sample, X, y, groups


cities = {}     # city -> (X, y, groups)
family = {}     # column -> group: size, location, host, reviews, ...

for city, snapshot in SNAPSHOTS.items():
    sample, X, y, groups = load_city(city, snapshot)
    cities[city] = (X, y, groups)

    fam = strfair.feature_families(sample)
    for col in X.columns:
        family[col] = fam[col] if col in fam.index else "sentiment"

    # nothing from LEAKY or DEMAND_RESPONSE may be left
    assert not set(X.columns) & set(strfair.LEAKY + strfair.DEMAND_RESPONSE), city

family = pd.Series(family)
print(len(family), "columns before screening")
family.value_counts()

/Users/krystal/Desktop/GeorgiaTech/Courses/26Fall/CSE6242_DVA/Project/cse6242-airbnb/strfair/features.py:323: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["price_per_person"] = out["price"] / out["accommodates"].where(out["accommodates"] > 0)


/Users/krystal/Desktop/GeorgiaTech/Courses/26Fall/CSE6242_DVA/Project/cse6242-airbnb/strfair/features.py:323: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["price_per_person"] = out["price"] / out["accommodates"].where(out["accommodates"] > 0)


/Users/krystal/Desktop/GeorgiaTech/Courses/26Fall/CSE6242_DVA/Project/cse6242-airbnb/strfair/features.py:323: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["price_per_person"] = out["price"] / out["accommodates"].where(out["accommodates"] > 0)


/Users/krystal/Desktop/GeorgiaTech/Courses/26Fall/CSE6242_DVA/Project/cse6242-airbnb/strfair/features.py:323: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["price_per_person"] = out["price"] / out["accommodates"].where(out["accommodates"] > 0)


/Users/krystal/Desktop/GeorgiaTech/Courses/26Fall/CSE6242_DVA/Project/cse6242-airbnb/strfair/features.py:323: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["price_per_person"] = out["price"] / out["accommodates"].where(out["accommodates"] > 0)


/Users/krystal/Desktop/GeorgiaTech/Courses/26Fall/CSE6242_DVA/Project/cse6242-airbnb/strfair/features.py:323: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["price_per_person"] = out["price"] / out["accommodates"].where(out["accommodates"] > 0)


84 columns before screening


amenities       18
reviews         14
host            12
listing_text    12
target_quote    11
size            10
location         6
availability     1
Name: count, dtype: int64

## A1. Data quality and leakage check

The first cell stops if any column has Spearman |ρ| of 0.9 or more with log price, which usually means it was
computed from the price. (discount_amount was dropped for this reason; discount_share and has_discount were
checked in notebook 05 and are fine.)

Columns where 95% or more of listings share one value in at least 4 cities are dropped. Missing values are
only shown here; both models fill them with the median.

In [3]:
# leakage check: stops if any column tracks log price almost perfectly
for city, (X, y, groups) in cities.items():
    rho = X.select_dtypes("number").corrwith(y, method="spearman").abs()
    rho = rho.dropna()                       # columns with one value in this city have no correlation
    assert (rho < 0.9).all(), f"{city}: check {list(rho[rho >= 0.9].index)} for leakage"

/opt/anaconda3/envs/airbnb-project/lib/python3.11/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


/opt/anaconda3/envs/airbnb-project/lib/python3.11/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


/opt/anaconda3/envs/airbnb-project/lib/python3.11/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/opt/anaconda3/envs/airbnb-project/lib/python3.11/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


/opt/anaconda3/envs/airbnb-project/lib/python3.11/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/opt/anaconda3/envs/airbnb-project/lib/python3.11/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


In [4]:
missing = {}
for city, (X, y, groups) in cities.items():
    missing[city] = X.isna().mean()
missing = pd.DataFrame(missing)

print("columns with more than 20% missing in some city:")
missing[missing.max(axis=1) > 0.2].round(2)

columns with more than 20% missing in some city:


,new-york-city,los-angeles,toronto,san-diego,clark-county-nv,jersey-city
months_since_last_review,0.26,0.24,0.18,0.15,0.27,0.16
review_scores_accuracy,0.26,0.24,0.18,0.15,0.27,0.16
review_scores_checkin,0.26,0.24,0.18,0.15,0.27,0.16
review_scores_cleanliness,0.26,0.24,0.18,0.15,0.27,0.16
review_scores_communication,0.26,0.24,0.18,0.15,0.27,0.16
review_scores_location,0.26,0.24,0.18,0.15,0.27,0.16
review_scores_rating,0.26,0.24,0.18,0.15,0.27,0.16


In [5]:
# share of the most common value in a column, e.g. 0.98 if 98% of listings have Wi-Fi
def most_common_share(column):
    shares = column.value_counts(normalize=True, dropna=False)   # share of each value, largest first
    return shares.iloc[0]


top_share = {}
for city, (X, y, groups) in cities.items():
    shares = {}
    for col in X.columns:
        shares[col] = most_common_share(X[col])
    top_share[city] = pd.Series(shares)
top_share = pd.DataFrame(top_share)     # one row per column, one column per city

# almost constant: one value covers 95%+ of listings in at least 4 cities
n_cities = (top_share >= 0.95).sum(axis=1)
almost_constant = n_cities[n_cities >= 4].index.tolist()
print("almost one value in 4+ cities:", almost_constant)
top_share.loc[almost_constant].round(3)

almost one value in 4+ cities: ['am_breakfast', 'am_smoking_allowed', 'am_wifi', 'amg_breakfast', 'amg_connectivity', 'has_description', 'has_neighborhood_overview', 'host_has_profile_pic', 'host_identity_verified']


,new-york-city,los-angeles,toronto,san-diego,clark-county-nv,jersey-city
am_breakfast,0.955,0.965,0.985,0.976,0.965,0.975
am_smoking_allowed,0.979,0.968,0.988,0.983,0.961,0.994
am_wifi,0.983,0.988,0.991,0.991,0.986,0.998
amg_breakfast,0.997,0.999,1.000,0.997,1.000,1.000
amg_connectivity,0.984,0.988,0.991,0.992,0.986,0.999
has_description,0.996,0.996,0.997,0.999,0.998,0.999
has_neighborhood_overview,1.000,1.000,1.000,1.000,1.000,1.000
host_has_profile_pic,0.956,0.967,0.958,0.980,0.959,0.967
host_identity_verified,0.984,0.987,0.983,0.973,0.983,0.980


Result: the leakage check passes everywhere. 9 columns are dropped for having almost one value: am_wifi,
amg_connectivity, am_breakfast, amg_breakfast, am_smoking_allowed, host_identity_verified,
host_has_profile_pic, has_description, has_neighborhood_overview. Only the review scores and
months_since_last_review have missing values (15-27%, the listings with no reviews).

## A2. Column types

checkin_dow and checkin_month are categories (Saturday isn't 5 times Monday), so they're converted to text.

In [6]:
for city, (X, y, groups) in cities.items():
    X["checkin_dow"] = X["checkin_dow"].astype(str)
    X["checkin_month"] = X["checkin_month"].astype(str)

categorical = sorted(set(strfair.CATEGORICAL + ["checkin_dow", "checkin_month"]) & set(family.index))
print("categorical columns:", categorical)

categorical columns: ['checkin_dow', 'checkin_month', 'lead_band', 'min_nights_band', 'neighbourhood', 'neighbourhood_group', 'property_group', 'room', 'room_type']


## A3. Shared candidate list

What's left after A1. Both models start from here.

In [7]:
shared_drop = set(almost_constant)
candidates = [c for c in family.index if c not in shared_drop]

pd.DataFrame({"column": candidates, "group": family[candidates].values}).to_csv(OUT / "candidates.csv", index=False)
print(f"{len(family)} columns -> {len(candidates)} candidates")
family[candidates].value_counts()

84 columns -> 75 candidates


reviews         14
amenities       13
target_quote    11
size            10
host            10
listing_text    10
location         6
availability     1
Name: count, dtype: int64

## B. Elastic net

Only Elastic net needs this, because its coefficients are what we report. If two columns carry the same
information, the weight gets split between them and the coefficients stop making sense. The forest doesn't
have that problem.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


def make_enet(X):
    cat = list(X.select_dtypes(exclude="number").columns)
    num = [c for c in X.columns if c not in cat]

    # numbers: fill missing values with the median, then scale
    # categories: one column per level, levels with < 20 listings are grouped together
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat),
    ])
    return make_pipeline(pre, ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000))


def make_rf(X):
    cat = list(X.select_dtypes(exclude="number").columns)
    num = [c for c in X.columns if c not in cat]

    # trees don't need scaling or one-hot columns: each category just gets a number
    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)), cat),
    ])
    # 2 cores per forest: the 5 folds run in parallel, 5 x 2 = 10 cores
    return make_pipeline(pre, RandomForestRegressor(n_estimators=100, min_samples_leaf=5,
                                                    n_jobs=2, random_state=42))


gkf = GroupKFold(n_splits=5)   # the folds only depend on the hosts, so every run uses the same folds


# a new model fit on each fold, MAE on the held-out fold; the 5 folds run in parallel
def run_model(make_model, X, y, groups):
    scores = cross_val_score(make_model(X), X, y, groups=groups, cv=gkf,
                             scoring="neg_mean_absolute_error", n_jobs=5)
    return -scores             # MAE of each fold

### B1. One form of each column

Keep one version of each column. A few exceptions, all because Elastic net did worse without them in NYC, LA
or San Diego: room_type, lead_days (lead_band stays too, as a quote control), the coordinates (a linear model
can still pick up "closer to the coast, more expensive"), and host_listings_count. That last one isn't a
duplicate: it counts the host's listings on all of Airbnb, while log_host_listings only counts this city.

| columns | keep |
|---|---|
| km_to_centre, log_km_to_centre | log_km_to_centre |
| number_of_reviews, log_reviews | log_reviews |
| calculated_host_listings_count, log_host_listings, multi_host, single_listing_host | log_host_listings |
| review_scores_rating, rating_filled | rating_filled |
| no_reviews, no_rating (the same listings) | no_rating |
| accommodates, guests_per_bedroom, guests_per_bed | accommodates |
| room_type, room | room_type |
| minimum_nights, min_nights_band | min_nights_band |

In [9]:
other_forms = ["km_to_centre", "number_of_reviews", "calculated_host_listings_count",
               "multi_host", "single_listing_host", "guests_per_bedroom", "guests_per_bed",
               "review_scores_rating", "no_reviews", "room", "minimum_nights"]

enet_cols = [c for c in candidates if c not in other_forms]
print(f"{len(candidates)} candidates -> {len(enet_cols)} after B1")

75 candidates -> 64 after B1


### B2. VIF

Drop the column with the highest VIF (median over the 6 cities), recompute, and repeat until nothing is above
10. Quote controls are never dropped. Numeric columns only.

In [10]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


# VIF of every column in X, computed in one city
def vif(X):
    num = X.fillna(X.median())               # VIF can't handle missing values
    num = num.loc[:, num.std() > 0]          # skip columns with only one value in this city
    num = add_constant(num)                  # VIF needs an intercept column ("const")

    values = {}
    for i, col in enumerate(num.columns):
        if col != "const":
            values[col] = variance_inflation_factor(num.values, i)
    return pd.Series(values)


numeric = [c for c in enet_cols if c not in categorical]
vif_removed = []

while True:
    table = {}
    for city, (X, y, groups) in cities.items():
        table[city] = vif(X[[c for c in numeric if c in X.columns]])
    median_vif = pd.DataFrame(table).median(axis=1)

    can_remove = median_vif.drop(QUOTE_CONTROLS, errors="ignore")
    worst = can_remove.idxmax()
    if can_remove[worst] <= 10:
        break
    print(f"remove {worst} (median VIF {can_remove[worst]:.1f})")
    vif_removed.append(worst)
    numeric.remove(worst)

enet_cols = [c for c in enet_cols if c not in vif_removed]
print(f"\n{len(enet_cols)} columns after B2. Highest median VIF left:")
median_vif.sort_values(ascending=False).head(10).round(1)


64 columns after B2. Highest median VIF left:


rating_filled             6.7
am_free_parking           6.2
log_reviews               5.7
amg_parking               5.6
accommodates              5.4
bedrooms                  5.1
is_monthly_quote          4.9
review_scores_accuracy    4.7
log_host_listings         4.4
beds                      4.2
dtype: float64

Result: nothing is above 10, so nothing is dropped. The highest is rating_filled at 6.7. accommodates and
log_host_listings were at 19 and 47 before B1 removed the columns derived from them; now they're 5.4 and 4.4.

### B3. Check: is the trimmed list as accurate as all candidates?

Elastic net with all candidates vs the trimmed list, on the same folds. The trimmed list passes if it isn't
worse by more than the std of the 5 fold differences.

In [11]:
rows = []
for city, (X, y, groups) in cities.items():
    cols = [c for c in enet_cols if c in X.columns]
    full = run_model(make_enet, X[[c for c in candidates if c in X.columns]], y, groups)
    trimmed = run_model(make_enet, X[cols], y, groups)
    diff = trimmed - full
    rows.append([city, len(cols), full.mean(), trimmed.mean(), diff.mean(), diff.mean() <= diff.std()])

enet_check = pd.DataFrame(rows, columns=["city", "columns", "all_candidates", "trimmed", "change", "ok"])
enet_check.to_csv(OUT / "enet_check.csv", index=False)
enet_check.round(4)

,city,columns,all_candidates,trimmed,change,ok
0,new-york-city,64,0.2943,0.2964,0.0021,False
1,los-angeles,64,0.2840,0.2842,0.0002,True
2,toronto,63,0.2503,0.2508,0.0005,True
3,san-diego,63,0.2837,0.2848,0.0011,True
4,clark-county-nv,63,0.3479,0.3501,0.0022,True
5,jersey-city,63,0.2232,0.2241,0.0009,True


Result: 64 columns (63 in cities without neighbourhood_group). MAE change vs all candidates:

| NYC | LA | Toronto | San Diego | Las Vegas | Jersey City |
|---|---|---|---|---|---|
| +0.0021 | +0.0002 | +0.0005 | +0.0011 | +0.0022 | +0.0009 |

Everything passes except NYC, and that's only +0.002 (under 1% of its MAE); NYC's folds are very close to each
other, so the bar is tight there. Keeping the raw minimum_nights didn't help: it moves with quote_nights
(Spearman 0.97) and VIF drops it.

## C. Random forest

Uses all shared candidates, including the coordinates and the other forms of each column.

In [12]:
rf_cols = list(candidates)

rows = []
for city, (X, y, groups) in cities.items():
    cols = [c for c in rf_cols if c in X.columns]
    rf_mae = run_model(make_rf, X[cols], y, groups)
    enet_mae = enet_check.loc[enet_check["city"] == city, "trimmed"].iloc[0]
    rows.append([city, len(cols), rf_mae.mean(), enet_mae, enet_mae - rf_mae.mean()])

rf_check = pd.DataFrame(rows, columns=["city", "columns", "rf_mae", "enet_trimmed_mae", "accuracy_given_up"])
rf_check.to_csv(OUT / "rf_check.csv", index=False)
rf_check.round(4)

,city,columns,rf_mae,enet_trimmed_mae,accuracy_given_up
0,new-york-city,75,0.2806,0.2964,0.0157
1,los-angeles,75,0.2698,0.2842,0.0145
2,toronto,74,0.2460,0.2508,0.0048
3,san-diego,74,0.2773,0.2848,0.0075
4,clark-county-nv,74,0.3087,0.3501,0.0413
5,jersey-city,74,0.2295,0.2241,-0.0054


Result: 75 columns (74 in four cities). MAE the interpretable model gives up vs the forest: 0.016 in NYC, 0.015
in LA, 0.005 in Toronto, 0.008 in San Diego and 0.041 in Las Vegas. In Jersey City Elastic net is 0.005 better.
Las Vegas has the biggest gap, probably because its location effect isn't linear and the forest can split on
the coordinates.

### C1. Permutation importance

Train on 4 folds, shuffle one column at a time in the held-out fold, and see how much MAE goes up. One fold per
city is enough for a ranking. This is only for explaining the forest, not for dropping columns. The forest's
built-in feature_importances_ isn't used: it's computed on the training data and favours columns with many
values.

In [13]:
from sklearn.inspection import permutation_importance

importance = {}
for city, (X, y, groups) in cities.items():
    Xr = X[[c for c in rf_cols if c in X.columns]]
    train_idx, test_idx = next(gkf.split(Xr, y, groups))
    model = make_rf(Xr).fit(Xr.iloc[train_idx], y.iloc[train_idx])
    result = permutation_importance(model, Xr.iloc[test_idx], y.iloc[test_idx],
                                    scoring="neg_mean_absolute_error", n_repeats=3, random_state=42)
    importance[city] = pd.Series(result.importances_mean, index=Xr.columns)
    print(city, "done")

importance = pd.DataFrame(importance)
importance["median"] = importance.median(axis=1)
importance = importance.sort_values("median", ascending=False)
importance.round(4).to_csv(OUT / "rf_permutation_importance.csv")
importance.head(20).round(4)

new-york-city done


los-angeles done


toronto done


san-diego done


clark-county-nv done


jersey-city done


,new-york-city,los-angeles,toronto,san-diego,clark-county-nv,jersey-city,median
quote_nights,0.1841,0.1258,0.2539,0.0422,0.1447,0.0875,0.1353
room_type,0.0733,0.0437,0.0930,0.0010,0.0011,0.0758,0.0585
baths,0.0258,0.0932,0.0440,0.0340,0.0431,0.0312,0.0386
accommodates,0.0385,0.0289,0.0299,0.1854,0.1858,0.0299,0.0342
longitude,0.0431,0.0665,0.0023,0.0338,0.0099,0.0138,0.0238
bedrooms,0.0426,0.1396,0.0187,0.0236,0.0055,0.0228,0.0232
shared_bath,0.0230,0.0080,0.0092,0.0078,0.0078,0.0024,0.0079
lead_days,0.0662,0.0049,0.0473,0.0029,0.0039,0.0101,0.0075
minimum_nights,0.0026,0.0269,0.0002,0.0107,0.0008,0.0388,0.0067
log_km_to_centre,0.0385,0.0058,0.0247,0.0034,0.0071,0.0003,0.0065


Result: quote_nights is far ahead (0.25 in Toronto, 0.18 in NYC, 0.15 in Las Vegas). Prices are per night for
one quote and monthly quotes are discounted, so quote length matters most, which is why the quote controls stay
in both models. Size comes next: room type (0.09 in Toronto, 0.08 in Jersey City, 0.07 in NYC), bathrooms, guests (0.19 in
San Diego and Las Vegas) and bedrooms (0.14 in LA). Then location: longitude (0.07 in LA) and distance to centre
(0.04 in NYC). Host, review, amenity and text columns are all 0.015 or less. Correlated columns split their
importance; km_to_centre and log_km_to_centre each get about 0.04 in NYC.

## Final lists

In [14]:
def reason(col):
    if col in almost_constant:
        return "almost one value (A1)"
    if col in other_forms:
        return "Elastic net: other form of a column (B1)"
    if col in vif_removed:
        return "Elastic net: VIF > 10 (B2)"
    return ""


rows = []
for col in family.index:
    rows.append([col, family[col], col in candidates, col in enet_cols, col in rf_cols, reason(col)])
decisions = pd.DataFrame(rows, columns=["column", "group", "candidate", "elastic_net", "random_forest", "reason"])
decisions.to_csv(OUT / "decisions.csv", index=False)

# copy these into strfair/features.py
print("CANDIDATES =", candidates)
print("ENET_FEATURES =", enet_cols)
print("RF_FEATURES =", rf_cols)
decisions[decisions["reason"] != ""]

CANDIDATES = ['lead_days', 'quote_nights', 'weekend_checkin', 'checkin_dow', 'checkin_month', 'is_monthly_quote', 'lead_band', 'has_discount', 'discount_share', 'minimum_nights', 'min_nights_band', 'room_type', 'room', 'property_group', 'accommodates', 'bedrooms', 'beds', 'shared_bath', 'baths', 'guests_per_bedroom', 'guests_per_bed', 'latitude', 'longitude', 'km_to_centre', 'log_km_to_centre', 'neighbourhood', 'neighbourhood_group', 'superhost', 'host_listings_count', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 'log_host_listings', 'multi_host', 'single_listing_host', 'host_entire_home_share', 'host_years', 'number_of_reviews', 'log_reviews', 'no_reviews', 'tenure_months', 'new_listing', 'months_since_last_review', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'no_rating', 'rating_filled'

,column,group,candidate,elastic_net,random_forest,reason
9,minimum_nights,target_quote,True,False,True,Elastic net: other form of a column (B1)
12,room,size,True,False,True,Elastic net: other form of a column (B1)
19,guests_per_bedroom,size,True,False,True,Elastic net: other form of a column (B1)
20,guests_per_bed,size,True,False,True,Elastic net: other form of a column (B1)
23,km_to_centre,location,True,False,True,Elastic net: other form of a column (B1)
28,host_identity_verified,host,False,False,False,almost one value (A1)
29,host_has_profile_pic,host,False,False,False,almost one value (A1)
31,calculated_host_listings_count,host,True,False,True,Elastic net: other form of a column (B1)
35,multi_host,host,True,False,True,Elastic net: other form of a column (B1)
36,single_listing_host,host,True,False,True,Elastic net: other form of a column (B1)


## Findings

2026-09 snapshots, without sentiment. 

- **Shared candidates:** 75 of 84 columns. Dropped: nine columns with almost one value (Wi-Fi, breakfast,
  smoking, verified ID, profile picture, description flags). 
- **Elastic net (64 columns):** one form of each column, plus `room_type`, `lead_days`, the coordinates and
  `host_listings_count`, which it needs in NYC, LA and San Diego. No VIF above 10. As accurate as all candidates
  in 5 of 6 cities; NYC is +0.002 (under 1% of its MAE).
- **Random forest (75 columns):** all shared candidates. It is 0.005–0.041 MAE better than Elastic net in 5
  cities (most in Las Vegas) and slightly worse in Jersey City.
- **What drives price for the forest:** quote length first, then size, then location. Host, reviews, amenities
  and listing text add little on their own.